# 2.1 Superseded design check: the curve-based critical-point label (2022)
The original design defined the event physically, as the onset of occupancy above an MFD-style
critical occupancy estimated from each district's 2022 flow-occupancy curve. Occupancy was grouped into
bins 0.5 points wide, the highest median flow among usable bins (at least 100 slots, consecutive from
the lowest bin) was the capacity flow, and the critical occupancy was the first bin whose median flow
reached 95% of it. A district was to be studied only if its curve showed a supported peak followed by
falling flow. These rules were fixed before any curve was drawn.

This notebook reproduces the finding reported in the paper (Section 3.4): no district passed, and the
label was replaced by the operational proxy built in `2.0-anh-label-construction.ipynb`. It is not part
of the final pipeline.

In [1]:
import numpy as np
import polars as pl

from madrid_congestion.config import DISTRICT_MINIMUM, PROCESSED_DATA_DIR
from madrid_congestion.districts import PANEL_PATH, VALID_READING, slot_rows

BIN_WIDTH = 0.5  # occupancy points per bin
MIN_BIN_SLOTS = 100  # a usable bin holds at least this many valid slots
CAPACITY_SHARE = 0.95  # critical occupancy: the first usable bin whose median flow reaches this share of capacity flow
PEAK_TAIL_BINS = 3  # eligibility: the flow peak must not lie in the last 3 usable bins
LAST_BIN_MAX = 0.90  # eligibility: flow in the last usable bin is at most 90% of the peak

series = pl.read_parquet(PROCESSED_DATA_DIR / "district-series-2022.parquet")
panel = pl.read_parquet(PANEL_PATH)
panel_sizes = panel.group_by("distrito").agg(panel_sensors=pl.len())
valid = series.filter(pl.col("valid_slot")).with_columns(
    bin_index=(pl.col("occupancy") / BIN_WIDTH).floor().cast(pl.Int32),  # bin lower edge = bin_index * BIN_WIDTH
)
pl.Config.set_tbl_rows(-1)
print(f"{valid.height:,} valid district slots of {series.height:,}")

617,514 valid district slots of 700,620


## 1. Curves and usable bins

In [2]:
def bin_curve(slots: pl.DataFrame) -> pl.DataFrame:
    # Per district and occupancy bin: the number of valid slots and their median flow.
    return (
        slots.group_by("distrito", "bin_index")
        .agg(slots=pl.len(), median_flow=pl.col("flow").median())
        .sort("distrito", "bin_index")
    )


def usable_bins(curve: pl.DataFrame) -> pl.DataFrame:
    # Bins from each district's lowest bin upwards, without gaps, stopping before the first bin with < MIN_BIN_SLOTS.
    return (
        curve.sort("distrito", "bin_index")
        .with_columns(
            in_run=(pl.col("bin_index") - pl.col("bin_index").min().over("distrito") == pl.int_range(pl.len()).over("distrito"))
            & (pl.col("slots") >= MIN_BIN_SLOTS)
        )
        .filter(pl.col("in_run").cast(pl.Int8).cum_min().over("distrito") == 1)
        .drop("in_run")
    )


curve = bin_curve(valid)
usable = usable_bins(curve)
usable.group_by("distrito").agg(usable_bins=pl.len(), top_edge=(pl.col("bin_index").max() + 1) * BIN_WIDTH).sort("distrito")

distrito,usable_bins,top_edge
i64,u32,f64
2,29,14.5
3,31,15.5
4,33,16.5
5,35,17.5
6,25,12.5
7,29,14.5
8,25,12.5
13,29,14.5
14,27,13.5


## 2. Capacity flow, critical occupancy and eligibility

In [3]:
def critical_points(usable: pl.DataFrame, share: float) -> pl.DataFrame:
    # Capacity flow = highest median flow among usable bins; critical occupancy = lower edge of the
    # first usable bin whose median flow reaches `share` of it.
    return (
        usable.with_columns(capacity_flow=pl.col("median_flow").max().over("distrito"))
        .filter(pl.col("median_flow") >= share * pl.col("capacity_flow"))
        .group_by("distrito")
        .agg(capacity_flow=pl.col("capacity_flow").first(), critical_occupancy=pl.col("bin_index").min() * BIN_WIDTH)
        .sort("distrito")
    )


def eligibility(usable: pl.DataFrame) -> pl.DataFrame:
    # The fixed eligibility rule. "Every used bin has >= 100 slots" holds by construction of the usable bins.
    shape = (
        usable.sort("distrito", "bin_index")
        .group_by("distrito", maintain_order=True)
        .agg(
            usable_bins=pl.len(),
            peak_position=pl.col("median_flow").arg_max(),
            peak_flow=pl.col("median_flow").max(),
            last_flow=pl.col("median_flow").last(),
        )
    )
    return (
        shape.join(panel_sizes, on="distrito")
        .with_columns(
            enough_sensors=pl.col("panel_sensors") >= DISTRICT_MINIMUM,
            peak_not_at_end=pl.col("peak_position") < pl.col("usable_bins") - PEAK_TAIL_BINS,
            falls_after_peak=pl.col("last_flow") <= LAST_BIN_MAX * pl.col("peak_flow"),
        )
        .with_columns(eligible=pl.col("enough_sensors") & pl.col("peak_not_at_end") & pl.col("falls_after_peak"))
    )


points = critical_points(usable, CAPACITY_SHARE)
summary = eligibility(usable).join(points, on="distrito").sort("distrito")
print(f"eligible: {summary['eligible'].sum()} of {summary.height} districts")
summary

eligible: 0 of 13 districts


distrito,usable_bins,peak_position,peak_flow,last_flow,panel_sensors,enough_sensors,peak_not_at_end,falls_after_peak,eligible,capacity_flow,critical_occupancy
i64,u32,u32,f64,f64,u32,bool,bool,bool,bool,f64,f64
2,29,28,905.74714,905.74714,83,true,false,false,false,905.74714,12.0
3,31,28,919.253333,917.027027,78,true,false,false,false,919.253333,13.0
4,33,31,1008.209677,1005.722222,128,true,false,false,false,1008.209677,13.5
5,35,34,910.352273,910.352273,181,true,false,false,false,910.352273,16.0
6,25,24,803.60759,803.60759,139,true,false,false,false,803.60759,11.0
7,29,28,986.376147,986.376147,113,true,false,false,false,986.376147,11.5
8,25,24,510.025641,510.025641,299,true,false,false,false,510.025641,11.5
13,29,28,563.043103,563.043103,128,true,false,false,false,563.043103,11.5
14,27,26,558.398077,558.398077,66,true,false,false,false,558.398077,11.5


**Result.** None of the 13 districts the procedure could evaluate passes. The highest median flow lies at
or close to the last usable bin, so no supported falling branch exists.

## 3. Districts the procedure could not evaluate

In [4]:
missing = sorted(set(panel_sizes["distrito"]) - set(usable["distrito"]))  # districts without a single usable bin
curve.filter(pl.col("distrito").is_in(missing)).group_by("distrito", maintain_order=True).agg(
    lowest_bin_slots=pl.col("slots").first(),
    first_dense_edge=pl.col("bin_index").filter(pl.col("slots") >= MIN_BIN_SLOTS).min() * BIN_WIDTH,
    valid_slots=pl.col("slots").sum(),
)

distrito,lowest_bin_slots,first_dense_edge,valid_slots
i64,u32,f64,u32
1,53,1.5,30244
9,1,0.5,29841
10,2,0.5,33505
11,1,0.5,33417
12,14,0.5,33614
17,46,0.5,33586
18,48,0.5,30056


**Result.** The other seven districts are inconclusive: their lowest bin holds 1–53 slots, fewer than
100, so the run of usable bins ends before it starts.

## 4. Integrity audit
Checks whether the zero-eligible result comes from an implementation or data problem. The rules were
not weakened to recover a falling branch.

In [5]:
duplicates = series.select(pl.struct("distrito", "slot_start").is_duplicated().sum()).item()
edges_ok = valid.select(
    ((pl.col("bin_index") * BIN_WIDTH <= pl.col("occupancy")) & (pl.col("occupancy") < (pl.col("bin_index") + 1) * BIN_WIDTH)).all()
).item()
counts_ok = curve["slots"].sum() == valid.height

check = valid.filter(pl.col("distrito") == 15)  # largest district, median recomputed with numpy
medians_ok = all(
    abs(float(np.median(check.filter(pl.col("bin_index") == b)["flow"].to_numpy())) - m) < 1e-9
    for b, m in curve.filter(pl.col("distrito") == 15).select("bin_index", "median_flow").iter_rows()
)

march = (  # district means recomputed from sensor readings for one month
    slot_rows("2022-03").join(panel.select("id", "distrito"), on="id").filter(VALID_READING)
    .group_by("distrito", "slot_start").agg(occupancy_check=pl.col("ocupacion").mean(), sensors_check=pl.len())
)
averaging = series.join(march, on=["distrito", "slot_start"]).select(
    max_occupancy_difference=(pl.col("occupancy") - pl.col("occupancy_check")).abs().max(),
    sensors_match=(pl.col("sensors") == pl.col("sensors_check")).all(),
)
print(f"duplicate district slots: {duplicates}, bin edges ok: {edges_ok}, counts ok: {counts_ok}, medians ok: {medians_ok}")
averaging

duplicate district slots: 0, bin edges ok: True, counts ok: True, medians ok: True


max_occupancy_difference,sensors_match
f64,bool
0.0,true


In [6]:
# Invalid slots still carry a partial mean when some sensors reported; it is noisier (fewer sensors), so read with care.
partial = series.filter(pl.col("sensors") > 0)
by_band = (
    partial.with_columns(band=(pl.col("occupancy") / 2.5).floor() * 2.5)  # 2.5-point bands, for a readable table
    .group_by("band").agg(slots=pl.len(), valid_share=pl.col("valid_slot").mean()).sort("band")
)
edge = usable.group_by("distrito").agg(top_edge=(pl.col("bin_index").max() + 1) * BIN_WIDTH)
above_edge = (
    partial.join(edge, on="distrito")
    .group_by("distrito")
    .agg(
        valid_share_all=pl.col("valid_slot").mean(),
        slots_above_edge=(pl.col("occupancy") >= pl.col("top_edge")).sum(),
        valid_share_above_edge=pl.col("valid_slot").filter(pl.col("occupancy") >= pl.col("top_edge")).mean(),
    )
    .sort("distrito")
)
display(by_band)
above_edge

band,slots,valid_share
f64,u32,f64
0.0,205561,0.837299
2.5,138228,0.899319
5.0,124562,0.925836
7.5,111738,0.943251
10.0,59622,0.949968
12.5,23481,0.954091
15.0,12718,0.959585
17.5,6269,0.967459
20.0,2522,0.961538


distrito,valid_share_all,slots_above_edge,valid_share_above_edge
i64,f64,u32,f64
2,0.851199,830,0.906024
3,0.96051,317,0.987382
4,0.975354,164,0.963415
5,0.945121,606,0.99505
6,0.965292,256,1.0
7,0.97079,371,1.0
8,0.905427,453,0.728477
13,0.977734,190,0.989474
14,0.944613,507,0.998028


**Result.** Every implementation check passes, and the March district means equal a recomputation from the
sensor readings. The valid share rises with occupancy up to 20% and falls only above 25%, where only 228
slots lie. Above each district's last usable bin it is at least as high as overall in 10 of the 12
studied districts; district 8 (73% against 91%) and district 4 (96% against 98%) are lower.